In [1]:
from pyspark.sql import SparkSession

import os
from pathlib import Path

from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd
import numpy as np


In [2]:
spark = SparkSession.builder \
        .master("local") \
        .appName("Brain Tumor Classification") \
        .getOrCreate()

bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
26/01/30 16:56:49 WARN Utils: Your hostname, LAPTOP-IVJGI2IE resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/01/30 16:56:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/30 16:56:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/30 16:56:50 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
def find_project_root():

    current = Path.cwd()

    for _ in range(5):
        if (current / ".git").exists():
            data_path = current / "data" / "raw"
            if data_path.exists():
                return data_path
        current = current.parent

    raise FileNotFoundError(
        "Projet root path could not be found"
    )

data_path = find_project_root()


In [4]:
data_path

PosixPath('/mnt/c/ESGI/4A/T2/Scala/MRI-Brain-Tumor-Classification/data/raw')

In [5]:
classes = sorted([d.name for d in data_path.iterdir() if d.is_dir()])
print(f"Nombre de classes: {len(classes)}")
print(f"\nClasses :\n{chr(10).join(classes)}")

Nombre de classes: 44

Classes :
Astrocitoma T1
Astrocitoma T1C+
Astrocitoma T2
Carcinoma T1
Carcinoma T1C+
Carcinoma T2
Ependimoma T1
Ependimoma T1C+
Ependimoma T2
Ganglioglioma T1
Ganglioglioma T1C+
Ganglioglioma T2
Germinoma T1
Germinoma T1C+
Germinoma T2
Glioblastoma T1
Glioblastoma T1C+
Glioblastoma T2
Granuloma T1
Granuloma T1C+
Granuloma T2
Meduloblastoma T1
Meduloblastoma T1C+
Meduloblastoma T2
Meningioma T1
Meningioma T1C+
Meningioma T2
Neurocitoma T1
Neurocitoma T1C+
Neurocitoma T2
Oligodendroglioma T1
Oligodendroglioma T1C+
Oligodendroglioma T2
Papiloma T1
Papiloma T1C+
Papiloma T2
Schwannoma T1
Schwannoma T1C+
Schwannoma T2
Tuberculoma T1
Tuberculoma T1C+
Tuberculoma T2
_NORMAL T1
_NORMAL T2


In [ ]:
class_counts = {}
total_images = 0

for class_name in classes:
    class_path = data_path / class_name
    count = sum(1 for pattern in ["*.jpeg", "*.JPG", "*.jpg"] for _ in class_path.glob(pattern))
    class_counts[class_name] = count
    total_images += count

df_counts = pd.DataFrame(list(class_counts.items()), columns=["Classe", "Nombre"])
df_counts = df_counts.sort_values("Nombre", ascending=False)

print(f"\nNombre total d'images : {total_images}")
print(f"\nDistribution par classe :\n{df_counts.to_string(index=False)}")


Nombre total d'images : 3225

Distribution par classe :
                Classe  Nombre
            _NORMAL T2     253
      Astrocitoma T1C+     185
            _NORMAL T1     180
       Schwannoma T1C+     158
       Meningioma T1C+     151
      Neurocitoma T1C+     147
         Meningioma T1     141
        Astrocitoma T2     131
        Astrocitoma T1     128
         Schwannoma T1     117
        Carcinoma T1C+     103
         Meningioma T2      99
        Neurocitoma T1      91
         Schwannoma T2      90
        Neurocitoma T2      90
     Glioblastoma T1C+      88
  Oligodendroglioma T1      86
      Tuberculoma T1C+      81
Oligodendroglioma T1C+      72
  Oligodendroglioma T2      66
          Carcinoma T2      64
          Carcinoma T1      61
   Meduloblastoma T1C+      59
       Glioblastoma T1      51
       Glioblastoma T2      50
           Papiloma T2      47
         Ependimoma T1      41
         Ependimoma T2      37
       Ependimoma T1C+      34
        Germi

In [ ]:
print(f"\n{'='*50}")
print(f"STATISTIQUES DE DISTRIBUTION")
print(f"{'='*50}")
print(f"Moyenne     : {df_counts["Nombre"].mean():.2f} images/classe")
print(f"Médiane     : {df_counts["Nombre"].median():.0f} images/classe")
print(f"Minimum     : {df_counts["Nombre"].min()} images")
print(f"Maximum     : {df_counts["Nombre"].max()} images")
print(f"Ecart-type  : {df_counts["Nombre"].std():.2f}")
print(f"{'='*50}\n")


STATISTIQUES DE DISTRIBUTION
Moyenne     : 73.30 images/classe
Médiane     : 60 images/classe
Minimum     : 6 images
Maximum     : 253 images
Ecart-type  : 56.60



In [ ]:
dimensions = []

for class_name in classes:
    class_path = data_path / class_name
    for pattern in ["*.jpeg", "*.JPG", "*.jpg"]:
        for img_path in class_path.glob(pattern):
            img = Image.open(img_path)
            dimensions.append(img.size)

df_dims = pd.DataFrame(dimensions, columns=["Largeur", "Hauteur"])
print(f"\nDimensions des images :")
print(df_dims.describe())


Dimensions des images :
           Largeur      Hauteur
count  3225.000000  3225.000000
mean    600.161860   626.975194
std      46.419076    17.444560
min     432.000000   473.000000
25%     571.000000   630.000000
50%     630.000000   630.000000
75%     630.000000   630.000000
max     630.000000   630.000000
